In [1]:
import os, shutil

def find_file(name):
    """Cari file di /kaggle/input tanpa masuk ke folder kompetisi (isinya ribuan DICOM)."""
    hasil = []
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d != "competitions"]
        if name in files:
            hasil.append(os.path.join(root, name))
    return hasil

for f in ["11_train_student_lat.py", "_meta.csv", "report_labels.csv", "knee_side.csv"]:
    print(f"{f:25s} -> {find_file(f)}")

src = find_file("11_train_student_lat.py")[0]
shutil.copy(src, "/kaggle/working/11_train_student_lat.py")
print("\nScript disalin dari:", src)

# pastikan alpha = 1.0 memang melewati label perbaikan (tanpa prediksi guru)
s = open("/kaggle/working/11_train_student_lat.py").read()
i = s.find('if CFG["alpha"] < 1.0')
print(s[i-80:i+250])

11_train_student_lat.py   -> ['/kaggle/input/notebooks/kellenvalerie/rsna-knee-23-dinov2-lat/11_train_student_lat.py']
_meta.csv                 -> ['/kaggle/input/notebooks/kellenvalerie/rsna-knee-02-preprocess-mri/vol_train/_meta.csv']
report_labels.csv         -> ['/kaggle/input/notebooks/kellenvalerie/rsna-knee-01d-label-ensemble/report_labels.csv']
knee_side.csv             -> ['/kaggle/input/notebooks/kellenvalerie/rsna-knee-22-sisi-lutut/knee_side.csv']

Script disalin dari: /kaggle/input/notebooks/kellenvalerie/rsna-knee-23-dinov2-lat/11_train_student_lat.py
kan: alpha * laporan + (1 - alpha) * guru OOF ----
    rep_ref = rep.copy()
    if CFG["alpha"] < 1.0:
        oof = rep[["StudyInstanceUID"]].merge(pd.read_csv(find_file(CFG["oof"])),
                                              on="StudyInstanceUID", how="left")
        Tm = oof[LABELS].values.astype(float)
        assert not


In [2]:
%%bash
cd /kaggle/working
python -u 11_train_student_lat.py --alpha 1.0 --meta _meta.csv \
    --backbone vit_small_patch14_dinov2.lvd142m --lr_backbone 1e-4 --llrd 0.8 \
    --batch 4 --epochs 12 --side_csv knee_side.csv --seed 42 --tag uji --debug 2>&1 | tail -n 25

Normalisasi sisi lutut: 2070 studi kiri dicerminkan | view [1, 2]
Train 64 | Val 32 | Gold holdout 58
Jumlah irisan per view: 16 | ukuran: 224 px | batch 4 x accum 1
Jumlah view per studi: 3
Backbone: vit_small_patch14_dinov2.lvd142m | tag: uji | parameter: 21.8 juta
LLRD 0.8: LR backbone 5.50e-06 (bawah) .. 1.00e-04 (atas)
/kaggle/working/11_train_student_lat.py:301: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()
  ep0 it0/16 loss 0.9364 (7.49 s/it)
Epoch 0: loss 0.7743 | val AUC 0.5194 | val-ref AUC 0.5281 | gold AUC 0.5417 | 0.5 menit

=== [uji] Checkpoint terbaik (val-ref AUC 0.5281) pada label e

In [3]:
%%bash
cd /kaggle/working
rm -f uji_* train_log_uji.csv val_preds_uji.csv gold_preds_uji.csv
COMMON="--alpha 1.0 --meta _meta.csv --backbone vit_small_patch14_dinov2.lvd142m
        --lr_backbone 1e-4 --llrd 0.8 --batch 4 --epochs 12 --side_csv knee_side.csv"

CUDA_VISIBLE_DEVICES=0 python -u 11_train_student_lat.py $COMMON --seed 42 --tag dinov2s_v1d_lat_noteacher \
    > log_dinov2s_v1d_lat_noteacher.txt 2>&1 &
CUDA_VISIBLE_DEVICES=1 python -u 11_train_student_lat.py $COMMON --seed 7 --tag dinov2s_v1d_lat_noteacher_s7 \
    > log_dinov2s_v1d_lat_noteacher_s7.txt 2>&1 &
wait
for t in dinov2s_v1d_lat_noteacher dinov2s_v1d_lat_noteacher_s7; do
  echo "===== $t ====="; tail -n 16 log_$t.txt
done
# baris "Label diperbaiki" seharusnya TIDAK muncul, karena guru tidak dipakai
grep -H "Label  :\|Label diperbaiki\|Normalisasi\|Train \|LLRD\|Error\|error" log_*.txt

===== dinov2s_v1d_lat_noteacher =====
Epoch 11: loss 0.3448 | val AUC 0.8569 | val-ref AUC 0.8292 | gold AUC 0.8481 | 16.5 menit

=== [dinov2s_v1d_lat_noteacher] Checkpoint terbaik (val-ref AUC 0.8292) pada label emas ===
ACL                AUC=0.902
MCL                AUC=0.944
Medial Meniscus    AUC=0.892
Lateral Meniscus   AUC=0.708
Medial OA          AUC=0.961
Lateral OA         AUC=0.779
PF OA              AUC=0.770
Effusion           AUC=0.884
Synovitis          AUC=0.695
Baker's            AUC=0.951
Contusion          AUC=0.869
Fracture           AUC=0.821
Mean AUC gold: 0.8481
===== dinov2s_v1d_lat_noteacher_s7 =====
Epoch 11: loss 0.3403 | val AUC 0.8562 | val-ref AUC 0.8265 | gold AUC 0.8593 | 15.4 menit

=== [dinov2s_v1d_lat_noteacher_s7] Checkpoint terbaik (val-ref AUC 0.8283) pada label emas ===
ACL                AUC=0.934
MCL                AUC=0.943
Medial Meniscus    AUC=0.875
Lateral Meniscus   AUC=0.773
Medial OA          AUC=0.969
Lateral OA         AUC=0.812
PF OA 

In [4]:
import pandas as pd
for tag in ["dinov2s_v1d_lat_noteacher", "dinov2s_v1d_lat_noteacher_s7"]:
    print(f"\n===== {tag} =====")
    print(pd.read_csv(f"/kaggle/working/train_log_{tag}.csv").round(4).to_string(index=False))

# angka model DENGAN guru dari notebook 23 (checkpoint terbaik)
dengan_guru = {"seed 42": (0.8820, 0.8631), "seed 7": (0.8780, 0.8624)}
print("\n===== Ringkasan (AUC validasi | AUC gold) =====")
for seed, tag in [("seed 42", "dinov2s_v1d_lat_noteacher"), ("seed 7", "dinov2s_v1d_lat_noteacher_s7")]:
    log = pd.read_csv(f"/kaggle/working/train_log_{tag}.csv")
    best = log.loc[log["val_ref_auc"].idxmax()]          # aturan checkpoint sama dengan script
    v0, g0 = dengan_guru[seed]
    print(f"{seed}: tanpa guru {best['val_auc']:.4f} | {best['gold_auc']:.4f}   "
          f"dengan guru {v0:.4f} | {g0:.4f}   selisih gold (dengan - tanpa) {g0 - best['gold_auc']:+.4f}")


===== dinov2s_v1d_lat_noteacher =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.5649   0.6913       0.6854    0.6758 16.2920
     1 0.5084   0.7532       0.7389    0.7611 16.0615
     2 0.4775   0.7842       0.7631    0.7998 16.0642
     3 0.4611   0.8127       0.7880    0.8137 16.0269
     4 0.4418   0.8343       0.8092    0.8317 16.1124
     5 0.4261   0.8431       0.8172    0.8456 16.1095
     6 0.4116   0.8497       0.8228    0.8514 16.1001
     7 0.3957   0.8574       0.8291    0.8612 16.1170
     8 0.3772   0.8567       0.8279    0.8469 16.0489
     9 0.3614   0.8562       0.8270    0.8479 16.0873
    10 0.3509   0.8566       0.8288    0.8476 16.0840
    11 0.3448   0.8569       0.8292    0.8481 16.5379

===== dinov2s_v1d_lat_noteacher_s7 =====
 epoch   loss  val_auc  val_ref_auc  gold_auc     min
     0 0.5616   0.7062       0.6953    0.7138 15.3894
     1 0.5034   0.7630       0.7412    0.7844 15.4210
     2 0.4739   0.8048       0.7748    0.8341 15.4469
 